# Семинар №2: Собираем быстрый пайплайн данных

Реализация всех 10 задач по построению эффективного видеопайплайна


In [2]:
# Установка всех необходимых библиотек
%pip install numpy torch torchvision av opencv-python pillow matplotlib tqdm

# Опционально: установка decord для аппаратного декодирования (GPU)
# Раскомментируйте следующую строку, если нужна поддержка GPU декодирования
%pip install decord
# или через conda:
# !conda install -c conda-forge decord

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 100.7 MB/s eta 0:00:00


In [3]:
import os
import time
import queue
import threading
import multiprocessing
import json
import hashlib
from collections import deque
from typing import List, Tuple, Optional, Callable
import numpy as np
import torch
import torch.nn as nn
import torch.utils.data as data
from torch.utils.data import Dataset, DataLoader
import av
import cv2
from PIL import Image
import torchvision.transforms.v2 as transforms_v2
import matplotlib.pyplot as plt
from tqdm import tqdm


In [4]:
# decord будет импортирован лениво только в задаче 7, чтобы избежать конфликта с av
DECORD_AVAILABLE = None  # Будет установлено при первом использовании


## Задача 1: Базовый декодер видеокадров


In [5]:
def check_cfr_vfr(filename: str) -> Tuple[str, dict]:
    """
    Проверяет, является ли видео CFR (Constant Frame Rate) или VFR (Variable Frame Rate).
    Основано на задании 1 из семинара 1.

    Returns:
        ('CFR' или 'VFR', словарь с метриками)
    """
    container = av.open(filename)
    video_stream = container.streams.video[0]

    # Получаем метаданные (используем правильные атрибуты PyAV)
    avg_frame_rate = video_stream.average_rate
    time_base = video_stream.time_base

    # Читаем первые 30 кадров для анализа интервалов
    pts_list = []
    frame_count = 0

    for frame in container.decode(video_stream):
        if frame.pts is not None:
            pts_list.append(frame.pts)
        frame_count += 1
        if frame_count >= 30:
            break

    container.close()

    # Анализируем интервалы между кадрами
    metrics = {
        'avg_frame_rate': float(avg_frame_rate) if avg_frame_rate else None,
        'time_base': float(time_base),
    }

    if len(pts_list) > 1:
        # Вычисляем интервалы в секундах
        intervals = [(pts_list[i+1] - pts_list[i]) * float(time_base)
                    for i in range(len(pts_list)-1)]
        metrics['intervals'] = intervals
        metrics['avg_interval'] = np.mean(intervals)
        metrics['std_interval'] = np.std(intervals)
        metrics['cv'] = metrics['std_interval'] / metrics['avg_interval'] if metrics['avg_interval'] > 0 else 0

        # Определяем CFR/VFR по коэффициенту вариации
        if metrics['cv'] > 0.1:
            result = 'VFR'
        else:
            result = 'CFR'
    else:
        result = 'Unknown'
        metrics['cv'] = None

    return result, metrics


In [6]:
def read_clip(filename: str, start: int = 0, num_frames: int = 16, stride: int = 2, verbose: bool = False) -> np.ndarray:
    """
    Читает клип из видео с помощью PyAV.
    Улучшенная версия с правильной обработкой временных меток (PTS).

    Args:
        filename: путь к видеофайлу
        start: начальный кадр (индекс)
        num_frames: количество кадров для чтения
        stride: шаг между кадрами
        verbose: выводить ли информацию о прочитанных кадрах

    Returns:
        numpy.ndarray формы (T, H, W, 3) в формате RGB
    """
    container = av.open(filename)
    video_stream = container.streams.video[0]
    total_frames = video_stream.frames if video_stream.frames else 0

    # Получаем параметры потока
    time_base = video_stream.time_base
    duration = float(video_stream.duration * time_base) if video_stream.duration else 0
    fps = float(video_stream.average_rate) if video_stream.average_rate else 30.0

    # Проверяем границы
    if total_frames > 0 and start >= total_frames:
        container.close()
        # Возвращаем пустой массив правильной формы (будет дополнен padding)
        return np.array([]).reshape(0, 1080, 1920, 3)

    frames = []
    frame_indices = []
    pts_list = []  # Список временных меток для анализа

    # Вычисляем целевые индексы кадров
    target_indices = [start + i * stride for i in range(num_frames)]

    # Переходим к начальному кадру используя правильный seek по времени
    # (подход из семинара 1, задание 2)
    try:
        if start > 0 and duration > 0:
            # Вычисляем время начала в секундах
            start_time = start / fps if fps > 0 else 0
            # Seek использует timestamp в базовых единицах времени потока
            seek_pts = int(start_time * time_base.denominator / time_base.numerator)
            container.seek(seek_pts, stream=video_stream)
    except Exception as e:
        if verbose:
            print(f"Warning: seek failed, starting from beginning: {e}")

    frame_count = 0
    last_pts = None

    # Читаем кадры с учетом временных меток
    for frame in container.decode(video_stream):
        current_pts = frame.pts

        # Проверяем, нужен ли нам этот кадр
        if frame_count in target_indices:
            if len(frames) < num_frames:
                # Конвертируем в RGB numpy array
                img = frame.to_ndarray(format='rgb24')
                frames.append(img)
                frame_indices.append(frame_count)
                pts_list.append(current_pts)
            else:
                break

        frame_count += 1

        # Останавливаемся если прошли все нужные кадры
        if len(frames) >= num_frames:
            break

        # Защита от бесконечного цикла
        if total_frames > 0 and frame_count >= total_frames:
            break

    container.close()

    if len(frames) == 0:
        # Возвращаем пустой массив (будет дополнен padding)
        return np.array([]).reshape(0, 1080, 1920, 3)

    result = np.stack(frames, axis=0)  # (T, H, W, 3)

    if verbose:
        print(f"Read {len(frames)} frames. Indices: {frame_indices}")
        print(f"Actual FPS: {fps:.2f}")
        if len(pts_list) > 1:
            # Анализ временных интервалов (для проверки CFR/VFR)
            intervals = [(pts_list[i+1] - pts_list[i]) * float(time_base)
                        for i in range(len(pts_list)-1)]
            if intervals:
                avg_interval = np.mean(intervals)
                std_interval = np.std(intervals)
                print(f"Frame intervals: avg={avg_interval:.4f}s, std={std_interval:.4f}s")
                if std_interval / avg_interval > 0.1:
                    print("  → VFR detected (variable frame rate)")
                else:
                    print("  → CFR detected (constant frame rate)")

    return result


## Задача 2: Реализация Dataset для видеоклипов


In [7]:
class VideoDataset(Dataset):
    """Dataset для загрузки видеоклипов."""

    def __init__(
        self,
        video_files: List[str],
        clip_len: int = 16,
        stride: int = 2,
        transform: Optional[Callable] = None
    ):
        self.video_files = video_files
        self.clip_len = clip_len
        self.stride = stride
        self.transform = transform

        # Предвычисляем количество клипов в каждом видео
        self.clip_counts = []
        for video_file in video_files:
            container = av.open(video_file)
            video_stream = container.streams.video[0]
            total_frames = video_stream.frames
            container.close()

            # Максимальный стартовый индекс для клипа
            max_start = max(0, total_frames - (clip_len * stride))
            num_clips = max(1, (max_start // stride) + 1)
            self.clip_counts.append(num_clips)

    def __len__(self):
        return sum(self.clip_counts)

    def __getitem__(self, idx):
        # Определяем, из какого видео брать клип
        video_idx = 0
        clip_idx = idx

        for i, count in enumerate(self.clip_counts):
            if clip_idx < count:
                video_idx = i
                break
            clip_idx -= count

        video_file = self.video_files[video_idx]

        # Вычисляем стартовый кадр
        start_frame = clip_idx * self.stride

        # Читаем клип
        clip = read_clip(video_file, start=start_frame, num_frames=self.clip_len, stride=self.stride, verbose=False)

        # Убеждаемся, что клип имеет правильную длину (padding если нужно)
        if len(clip) == 0:
            # Если не удалось прочитать кадры, пропускаем этот клип
            # Возвращаем нулевой тензор правильной формы
            if self.transform:
                # Создаем нулевой тензор формы (T, C, H, W)
                clip = torch.zeros(self.clip_len, 3, 224, 224, dtype=torch.float32)
            else:
                # Создаем нулевой numpy array и конвертируем
                clip = np.zeros((self.clip_len, 1080, 1920, 3), dtype=np.uint8)
        elif len(clip) < self.clip_len:
            # Дополняем последним кадром
            if len(clip) > 0:
                last_frame = clip[-1]
                padding_frames = [last_frame] * (self.clip_len - len(clip))
                clip = np.concatenate([clip, np.stack(padding_frames, axis=0)], axis=0)
            else:
                # Если нет кадров, создаем нулевой массив
                clip = np.zeros((self.clip_len, clip.shape[1] if len(clip.shape) > 1 else 1080,
                               clip.shape[2] if len(clip.shape) > 2 else 1920, 3), dtype=np.uint8)

        # Обрезаем если больше нужного
        clip = clip[:self.clip_len]

        # Применяем трансформации
        if self.transform:
            # Transform ожидает 3D (H, W, C), а клип 4D (T, H, W, C)
            # Применяем transform к каждому кадру отдельно
            transformed_frames = []
            for frame in clip:
                # frame имеет форму (H, W, 3)
                frame_tensor = self.transform(frame)
                transformed_frames.append(frame_tensor)
            # Собираем обратно в тензор формы (T, C, H, W)
            clip = torch.stack(transformed_frames, dim=0)
        else:
            # Если нет transform, конвертируем в тензор
            clip = torch.from_numpy(clip).permute(0, 3, 1, 2).float()  # (T, H, W, 3) -> (T, C, H, W)

        return clip


In [8]:
def measure_throughput(dataloader: DataLoader, num_iterations: int = 10) -> Tuple[float, float]:
    """
    Измеряет throughput (кадров/с) для DataLoader.

    Returns:
        (mean_fps, std_fps)
    """
    times = []
    frame_counts = []

    # Вычисляем общее количество батчей для progress bar
    total_batches = len(dataloader)

    for iteration in tqdm(range(num_iterations), desc="  Iterations", leave=False):
        start_time = time.time()
        frames_processed = 0

        for batch in tqdm(dataloader, desc=f"    Epoch {iteration+1}/{num_iterations}",
                         total=total_batches, leave=False, unit="batch"):
            frames_processed += batch.shape[0] * batch.shape[1]  # batch_size * clip_len

        elapsed = time.time() - start_time
        fps = frames_processed / elapsed if elapsed > 0 else 0

        times.append(elapsed)
        frame_counts.append(frames_processed)

    total_frames = sum(frame_counts)
    total_time = sum(times)
    mean_fps = total_frames / total_time if total_time > 0 else 0

    return mean_fps, np.std([fc / t for fc, t in zip(frame_counts, times)])


In [9]:
def task3_parallel_loading(video_files: List[str], output_dir: str = "HW1/results", force_rerun: bool = False):
    """Задача 3: Измерение throughput при разных num_workers.

    Args:
        video_files: список путей к видеофайлам
        output_dir: директория для сохранения результатов
        force_rerun: если True, перезапускает измерения даже если есть сохраненные результаты
    """
    os.makedirs(output_dir, exist_ok=True)

    # Создаем уникальный ключ для чекпоинта на основе параметров
    config_hash = hashlib.md5(
        (str(sorted(video_files)) + "16_2_4_5").encode()
    ).hexdigest()[:8]
    checkpoint_file = f"{output_dir}/task3_checkpoint_{config_hash}.json"

    # Пытаемся загрузить сохраненные результаты
    if not force_rerun and os.path.exists(checkpoint_file):
        try:
            with open(checkpoint_file, 'r') as f:
                saved_data = json.load(f)
                results = [(r['workers'], r['mean_fps'], r['std_fps']) for r in saved_data['results']]
                print(f"✓ Loaded checkpoint from {checkpoint_file}")
                print(f"  Previous results: {len(results)} configurations")

                # Проверяем, что все конфигурации есть
                num_workers_list = [1, 2, 4, 8]
                if len(results) == len(num_workers_list):
                    # Выводим результаты и строим график
                    workers, fps_values, _ = zip(*results)

                    plt.figure(figsize=(10, 6))
                    plt.plot(workers, fps_values, 'o-', linewidth=2, markersize=8)
                    plt.xlabel('Number of Workers', fontsize=12)
                    plt.ylabel('Throughput (FPS)', fontsize=12)
                    plt.title('Throughput vs Number of Workers', fontsize=14)
                    plt.grid(True, alpha=0.3)
                    plt.savefig(f"{output_dir}/task3_throughput.png", dpi=150, bbox_inches='tight')
                    plt.close()

                    print(f"\nResults saved to {output_dir}/task3_throughput.png")

                    max_fps = max(fps_values)
                    saturation_workers = next((w for w, f in zip(workers, fps_values) if f >= 0.95 * max_fps), workers[-1])
                    print(f"Saturation point: {saturation_workers} workers")

                    return results
        except Exception as e:
            print(f"Warning: Failed to load checkpoint: {e}. Running fresh measurements...")

    # Выполняем измерения
    transform = transforms_v2.Compose([
        transforms_v2.ToImage(),
        transforms_v2.Resize((224, 224)),
        transforms_v2.ToDtype(torch.float32, scale=True),
        transforms_v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    dataset = VideoDataset(video_files, clip_len=16, stride=2, transform=transform)

    num_workers_list = [1, 2, 4, 8]
    results = []

    for num_workers in tqdm(num_workers_list, desc="Testing num_workers", unit="config"):
        dataloader = DataLoader(
            dataset,
            batch_size=4,
            num_workers=num_workers,
            shuffle=False
        )

        print(f"\nTesting with num_workers={num_workers}...")
        mean_fps, std_fps = measure_throughput(dataloader, num_iterations=5)
        results.append((num_workers, mean_fps, std_fps))
        print(f"  ✓ Mean FPS: {mean_fps:.2f} ± {std_fps:.2f}")

        # Сохраняем промежуточные результаты после каждой конфигурации
        checkpoint_data = {
            'results': [
                {'workers': w, 'mean_fps': float(fps), 'std_fps': float(std)}
                for w, fps, std in results
            ],
            'timestamp': time.time()
        }
        with open(checkpoint_file, 'w') as f:
            json.dump(checkpoint_data, f, indent=2)

    # Построение графика
    workers, fps_values, _ = zip(*results)

    plt.figure(figsize=(10, 6))
    plt.plot(workers, fps_values, 'o-', linewidth=2, markersize=8)
    plt.xlabel('Number of Workers', fontsize=12)
    plt.ylabel('Throughput (FPS)', fontsize=12)
    plt.title('Throughput vs Number of Workers', fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.savefig(f"{output_dir}/task3_throughput.png", dpi=150, bbox_inches='tight')
    plt.close()

    print(f"\nResults saved to {output_dir}/task3_throughput.png")
    print(f"Checkpoint saved to {checkpoint_file}")

    # Определение точки насыщения
    max_fps = max(fps_values)
    saturation_workers = next((w for w, f in zip(workers, fps_values) if f >= 0.95 * max_fps), workers[-1])
    print(f"Saturation point: {saturation_workers} workers")

    return results


## Задача 4: Профилирование этапов пайплайна


In [10]:
def task4_profiling(video_files: List[str], output_dir: str = "HW1/results"):
    """Задача 4: Профилирование с torch.profiler."""
    os.makedirs(output_dir, exist_ok=True)

    transform = transforms_v2.Compose([
        transforms_v2.ToImage(),
        transforms_v2.Resize((224, 224)),
        transforms_v2.ToDtype(torch.float32, scale=True),
        transforms_v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    dataset = VideoDataset(video_files, clip_len=16, stride=2, transform=transform)
    dataloader = DataLoader(dataset, batch_size=4, num_workers=4, shuffle=False)

    # Заглушка модели
    model = lambda x: x.mean()

    # Профилирование
    with torch.profiler.profile(
        activities=[
            torch.profiler.ProfilerActivity.CPU,
            torch.profiler.ProfilerActivity.CUDA if torch.cuda.is_available() else None
        ],
        schedule=torch.profiler.schedule(wait=1, warmup=1, active=3, repeat=2),
        on_trace_ready=torch.profiler.tensorboard_trace_handler(f"{output_dir}/profiler"),
        record_shapes=True,
        with_stack=True
    ) as prof:
        for i, batch in enumerate(dataloader):
            if isinstance(batch, torch.Tensor):
                batch = batch.to('cuda' if torch.cuda.is_available() else 'cpu')
            result = model(batch)
            prof.step()

            if i >= 10:  # Ограничиваем количество итераций
                break

    print(f"Profiling results saved to {output_dir}/profiler")
    print("View with: tensorboard --logdir=" + output_dir + "/profiler")

    # Анализ соотношения времени
    key_averages = prof.key_averages()

    decode_time = 0
    prep_time = 0
    infer_time = 0

    for event in key_averages:
        name = event.key.lower()
        # cpu_time_total уже в микросекундах, делим на 1000 для миллисекунд
        cpu_time_ms = event.cpu_time_total / 1000 if hasattr(event, 'cpu_time_total') else 0

        if 'read_clip' in name or 'decode' in name:
            decode_time += cpu_time_ms
        elif 'transform' in name or 'resize' in name or 'normalize' in name:
            prep_time += cpu_time_ms
        elif 'mean' in name or 'model' in name:
            infer_time += cpu_time_ms

    total_time = decode_time + prep_time + infer_time
    if total_time > 0:
        print(f"\nTime distribution:")
        print(f"  Decode: {decode_time:.2f} ms ({100*decode_time/total_time:.1f}%)")
        print(f"  Preprocess: {prep_time:.2f} ms ({100*prep_time/total_time:.1f}%)")
        print(f"  Infer: {infer_time:.2f} ms ({100*infer_time/total_time:.1f}%)")
        print(f"\nRatio L_dec:L_prep:L_inf = {decode_time:.2f}:{prep_time:.2f}:{infer_time:.2f}")


## Задача 5: Prefetch и pinned memory


In [11]:
def task5_prefetch_pinned_memory(video_files: List[str], output_dir: str = "HW1/results", force_rerun: bool = False):
    """Задача 5: Сравнение с/без prefetch и pinned memory.

    Args:
        video_files: список путей к видеофайлам
        output_dir: директория для сохранения результатов
        force_rerun: если True, перезапускает измерения даже если есть сохраненные результаты
    """
    os.makedirs(output_dir, exist_ok=True)

    # Создаем уникальный ключ для чекпоинта
    config_hash = hashlib.md5(
        (str(sorted(video_files)) + "16_2_4_10").encode()
    ).hexdigest()[:8]
    checkpoint_file = f"{output_dir}/task5_checkpoint_{config_hash}.json"

    # Пытаемся загрузить сохраненные результаты
    if not force_rerun and os.path.exists(checkpoint_file):
        try:
            with open(checkpoint_file, 'r') as f:
                saved_data = json.load(f)
                results = [(r['name'], r['mean_fps'], r['std_fps'], r['jitter']) for r in saved_data['results']]
                print(f"✓ Loaded checkpoint from {checkpoint_file}")
                print(f"  Previous results: {len(results)} configurations")

                # Выводим результаты
                print("\n" + "="*60)
                print("Comparison Results:")
                print("="*60)
                for name, mean_fps, std_fps, jitter in results:
                    print(f"{name:20s} | FPS: {mean_fps:7.2f} ± {std_fps:6.2f} | Jitter: {jitter:.4f}")

                return results
        except Exception as e:
            print(f"Warning: Failed to load checkpoint: {e}. Running fresh measurements...")

    transform = transforms_v2.Compose([
        transforms_v2.ToImage(),
        transforms_v2.Resize((224, 224)),
        transforms_v2.ToDtype(torch.float32, scale=True),
        transforms_v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    dataset = VideoDataset(video_files, clip_len=16, stride=2, transform=transform)

    configs = [
        ("Baseline", {"prefetch_factor": None, "pin_memory": False}),
        ("Prefetch", {"prefetch_factor": 2, "pin_memory": False}),
        ("Pinned Memory", {"prefetch_factor": None, "pin_memory": True}),
        ("Both", {"prefetch_factor": 2, "pin_memory": True}),
    ]

    results = []

    for name, config in tqdm(configs, desc="Testing configurations", unit="config"):
        dataloader = DataLoader(
            dataset,
            batch_size=4,
            num_workers=4,
            shuffle=False,
            **config
        )

        print(f"\nTesting {name}...")

        fps_list = []
        latencies = []

        for iteration in tqdm(range(10), desc=f"  {name} iterations", leave=False):
            start = time.time()
            for batch in dataloader:
                if torch.cuda.is_available() and config.get("pin_memory", False):
                    batch = batch.to('cuda', non_blocking=True)
                pass  # Просто загружаем данные
            elapsed = time.time() - start
            fps_list.append(len(dataset) * 16 / elapsed)  # примерный FPS
            latencies.append(elapsed)

        mean_fps = np.mean(fps_list)
        std_fps = np.std(fps_list)
        jitter = np.std(latencies) / np.mean(latencies) if np.mean(latencies) > 0 else 0

        results.append((name, mean_fps, std_fps, jitter))
        print(f"  Mean FPS: {mean_fps:.2f} ± {std_fps:.2f}")
        print(f"  Jitter: {jitter:.4f}")

        # Сохраняем промежуточные результаты
        checkpoint_data = {
            'results': [
                {'name': n, 'mean_fps': float(fps), 'std_fps': float(std), 'jitter': float(j)}
                for n, fps, std, j in results
            ],
            'timestamp': time.time()
        }
        with open(checkpoint_file, 'w') as f:
            json.dump(checkpoint_data, f, indent=2)

    # Вывод результатов
    print("\n" + "="*60)
    print("Comparison Results:")
    print("="*60)
    for name, mean_fps, std_fps, jitter in results:
        print(f"{name:20s} | FPS: {mean_fps:7.2f} ± {std_fps:6.2f} | Jitter: {jitter:.4f}")

    print(f"\nCheckpoint saved to {checkpoint_file}")
    return results


## Задача 6: Pipeline overlap


In [12]:
class PipelineOverlap:
    """Реализация перекрытия декодирования и инференса."""

    def __init__(self, video_file: str, model: Callable, batch_size: int = 4):
        self.video_file = video_file
        self.model = model
        self.batch_size = batch_size
        self.frame_queue = queue.Queue(maxsize=10)
        self.result_queue = queue.Queue()

    def decode_thread(self):
        """Поток декодирования."""
        container = av.open(self.video_file)
        video_stream = container.streams.video[0]

        frame_count = 0
        batch = []

        for frame in container.decode(video_stream):
            img = frame.to_ndarray(format='rgb24')
            batch.append(img)

            if len(batch) >= self.batch_size:
                self.frame_queue.put(np.stack(batch))
                batch = []

            frame_count += 1
            if frame_count >= 100:  # Ограничение для теста
                break

        if batch:
            self.frame_queue.put(np.stack(batch))

        self.frame_queue.put(None)  # Сигнал окончания
        container.close()

    def infer_thread(self):
        """Поток инференса."""
        while True:
            batch = self.frame_queue.get()
            if batch is None:
                self.result_queue.put(None)
                break

            # Имитация инференса
            result = self.model(torch.from_numpy(batch).float())
            self.result_queue.put(result)

    def run_sequential(self):
        """Последовательное выполнение."""
        start = time.time()

        container = av.open(self.video_file)
        video_stream = container.streams.video[0]

        frame_count = 0
        for frame in container.decode(video_stream):
            img = frame.to_ndarray(format='rgb24')
            batch_tensor = torch.from_numpy(img).float().unsqueeze(0)
            result = self.model(batch_tensor)
            frame_count += 1
            if frame_count >= 100:
                break

        container.close()
        elapsed = time.time() - start
        return elapsed / frame_count if frame_count > 0 else 0

    def run_overlapped(self):
        """Перекрытое выполнение."""
        start = time.time()

        decode_thread = threading.Thread(target=self.decode_thread)
        infer_thread = threading.Thread(target=self.infer_thread)

        decode_thread.start()
        infer_thread.start()

        results = []
        while True:
            result = self.result_queue.get()
            if result is None:
                break
            results.append(result)

        decode_thread.join()
        infer_thread.join()

        elapsed = time.time() - start
        return elapsed / len(results) if results else 0


In [13]:
def task6_pipeline_overlap(video_file: str, output_dir: str = "HW1/results"):
    """Задача 6: Сравнение последовательного и перекрытого выполнения."""
    os.makedirs(output_dir, exist_ok=True)

    model = lambda x: x.mean()
    pipeline = PipelineOverlap(video_file, model)

    print("Running sequential pipeline...")
    seq_latency = pipeline.run_sequential()
    print(f"  Average latency: {seq_latency*1000:.2f} ms")

    print("Running overlapped pipeline...")
    overlap_latency = pipeline.run_overlapped()
    print(f"  Average latency: {overlap_latency*1000:.2f} ms")

    speedup = seq_latency / overlap_latency if overlap_latency > 0 else 0
    print(f"\nSpeedup: {speedup:.2f}x")

    return seq_latency, overlap_latency, speedup


## Задача 7: Аппаратное декодирование


In [14]:
def read_clip_decord(filename: str, start: int = 0, num_frames: int = 16, stride: int = 2, gpu: bool = False) -> np.ndarray:
    """Чтение клипа с помощью decord (GPU/CPU). Ленивый импорт для избежания конфликта с av."""
    global DECORD_AVAILABLE

    # Ленивый импорт decord только когда он действительно нужен
    if DECORD_AVAILABLE is None:
        try:
            import decord
            DECORD_AVAILABLE = True
        except ImportError:
            DECORD_AVAILABLE = False
            raise ImportError("decord not available. Install with: pip install decord or conda install -c conda-forge decord")

    if not DECORD_AVAILABLE:
        raise ImportError("decord not available")

    import decord  # Импортируем здесь, чтобы избежать конфликта при загрузке модуля

    ctx = decord.gpu(0) if gpu else decord.cpu(0)
    vr = decord.VideoReader(filename, ctx=ctx)

    frame_indices = [start + i * stride for i in range(num_frames)]
    frames = vr.get_batch(frame_indices).asnumpy()

    return frames


In [15]:
def task7_hardware_decoding(video_file: str, output_dir: str = "HW1/results"):
    """Задача 7: Сравнение PyAV (CPU) и decord (GPU)."""
    os.makedirs(output_dir, exist_ok=True)

    num_frames = 100
    results = []

    # PyAV (CPU)
    print("Testing PyAV (CPU)...")
    times = []
    for _ in range(5):
        start = time.time()
        read_clip(video_file, start=0, num_frames=num_frames, stride=1)
        times.append((time.time() - start) * 1000)  # мс

    avg_time_pyav = np.mean(times)
    fps_pyav = num_frames / (avg_time_pyav / 1000)
    results.append(("PyAV (CPU)", avg_time_pyav, fps_pyav))
    print(f"  Average time: {avg_time_pyav:.2f} ms")
    print(f"  FPS: {fps_pyav:.2f}")

    # decord (CPU) - ленивый импорт
    global DECORD_AVAILABLE
    if DECORD_AVAILABLE is None:
        try:
            import decord
            DECORD_AVAILABLE = True
        except ImportError:
            DECORD_AVAILABLE = False

    if DECORD_AVAILABLE:
        print("\nTesting decord (CPU)...")
        times = []
        for _ in range(5):
            start = time.time()
            read_clip_decord(video_file, start=0, num_frames=num_frames, stride=1, gpu=False)
            times.append((time.time() - start) * 1000)

        avg_time_decord_cpu = np.mean(times)
        fps_decord_cpu = num_frames / (avg_time_decord_cpu / 1000)
        results.append(("decord (CPU)", avg_time_decord_cpu, fps_decord_cpu))
        print(f"  Average time: {avg_time_decord_cpu:.2f} ms")
        print(f"  FPS: {fps_decord_cpu:.2f}")

        # decord (GPU) - если доступно
        if torch.cuda.is_available():
            print("\nTesting decord (GPU)...")
            times = []
            for _ in range(5):
                start = time.time()
                read_clip_decord(video_file, start=0, num_frames=num_frames, stride=1, gpu=True)
                times.append((time.time() - start) * 1000)

            avg_time_decord_gpu = np.mean(times)
            fps_decord_gpu = num_frames / (avg_time_decord_gpu / 1000)
            results.append(("decord (GPU)", avg_time_decord_gpu, fps_decord_gpu))
            print(f"  Average time: {avg_time_decord_gpu:.2f} ms")
            print(f"  FPS: {fps_decord_gpu:.2f}")

    # Вывод таблицы
    print("\n" + "="*60)
    print("Decoding Performance Comparison:")
    print("="*60)
    print(f"{'Method':<20} | {'Time (ms)':<15} | {'FPS':<10}")
    print("-" * 60)
    for method, avg_time, fps in results:
        print(f"{method:<20} | {avg_time:>13.2f} | {fps:>8.2f}")

    return results


## Задача 8: Оптимизация препроцессинга


In [16]:
def task8_gpu_preprocessing(video_files: List[str], output_dir: str = "HW1/results"):
    """Задача 8: Сравнение CPU и GPU препроцессинга."""
    os.makedirs(output_dir, exist_ok=True)

    video_file = video_files[0]  # Используем первое видео

    # CPU препроцессинг
    cpu_transform = transforms_v2.Compose([
        transforms_v2.ToImage(),
        transforms_v2.Resize((224, 224)),
        transforms_v2.ToDtype(torch.float32, scale=True),
        transforms_v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # GPU препроцессинг
    gpu_transform = transforms_v2.Compose([
        transforms_v2.ToImage(),
        transforms_v2.Resize((224, 224), antialias=True),
        transforms_v2.ToDtype(torch.float32, scale=True),
        transforms_v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    model = lambda x: x.mean()

    # CPU вариант
    print("Testing CPU preprocessing...")
    decode_times = []
    prep_times = []
    infer_times = []

    for i in range(10):
        # Декодирование
        decode_start = time.time()
        clip = read_clip(video_file, start=i*32, num_frames=16, stride=2, verbose=False)
        if len(clip) == 0:
            continue
        decode_time = time.time() - decode_start

        # Препроцессинг
        prep_start = time.time()
        # Применяем transform к каждому кадру (clip имеет форму (T, H, W, 3))
        processed_frames = []
        for frame_idx in range(clip.shape[0]):
            frame = clip[frame_idx]  # (H, W, 3) numpy array uint8
            processed_frame = cpu_transform(frame)
            processed_frames.append(processed_frame)
        processed = torch.stack(processed_frames, dim=0)  # (T, C, H, W)
        prep_time = time.time() - prep_start

        # Инференс
        infer_start = time.time()
        result = model(processed)
        infer_time = time.time() - infer_start

        decode_times.append(decode_time * 1000)
        prep_times.append(prep_time * 1000)
        infer_times.append(infer_time * 1000)

    cpu_decode = np.mean(decode_times)
    cpu_prep = np.mean(prep_times)
    cpu_infer = np.mean(infer_times)

    print(f"  Decode: {cpu_decode:.2f} ms")
    print(f"  Preprocess: {cpu_prep:.2f} ms")
    print(f"  Infer: {cpu_infer:.2f} ms")

    # GPU вариант
    device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
    if device != 'cpu':
        print(f"\nTesting GPU preprocessing (device: {device})...")
        decode_times_gpu = []
        prep_times_gpu = []
        infer_times_gpu = []

        for i in range(10):
            # Декодирование
            decode_start = time.time()
            clip = read_clip(video_file, start=i*32, num_frames=16, stride=2, verbose=False)
            if len(clip) == 0:
                continue
            decode_time = time.time() - decode_start

            # Препроцессинг
            prep_start = time.time()
            # Применяем transform к каждому кадру (clip имеет форму (T, H, W, 3))
            processed_frames = []
            for frame_idx in range(clip.shape[0]):
                frame = clip[frame_idx]  # (H, W, 3) numpy array uint8
                processed_frame = gpu_transform(frame)
                if device == 'cuda':
                    processed_frame = processed_frame.cuda()
                elif device == 'mps':
                    processed_frame = processed_frame.to('mps')
                processed_frames.append(processed_frame)
            processed = torch.stack(processed_frames, dim=0)  # (T, C, H, W)
            prep_time = time.time() - prep_start

            # Инференс
            infer_start = time.time()
            result = model(processed)
            infer_time = time.time() - infer_start

            decode_times_gpu.append(decode_time * 1000)
            prep_times_gpu.append(prep_time * 1000)
            infer_times_gpu.append(infer_time * 1000)

        gpu_decode = np.mean(decode_times_gpu)
        gpu_prep = np.mean(prep_times_gpu)
        gpu_infer = np.mean(infer_times_gpu)

        print(f"  Decode: {gpu_decode:.2f} ms")
        print(f"  Preprocess: {gpu_prep:.2f} ms")
        print(f"  Infer: {gpu_infer:.2f} ms")

        # График
        categories = ['Decode', 'Preprocess', 'Infer']
        cpu_times = [cpu_decode, cpu_prep, cpu_infer]
        gpu_times = [gpu_decode, gpu_prep, gpu_infer]

        x = np.arange(len(categories))
        width = 0.35

        fig, ax = plt.subplots(figsize=(10, 6))
        bars1 = ax.bar(x - width/2, cpu_times, width, label='CPU', alpha=0.8)
        bars2 = ax.bar(x + width/2, gpu_times, width, label='GPU', alpha=0.8)

        ax.set_ylabel('Time (ms)', fontsize=12)
        ax.set_title('CPU vs GPU Pipeline Stages', fontsize=14)
        ax.set_xticks(x)
        ax.set_xticklabels(categories)
        ax.legend()
        ax.grid(True, alpha=0.3, axis='y')

        plt.savefig(f"{output_dir}/task8_gpu_preprocessing.png", dpi=150, bbox_inches='tight')
        plt.close()

        print(f"\nResults saved to {output_dir}/task8_gpu_preprocessing.png")

        return {
            'cpu': (cpu_decode, cpu_prep, cpu_infer),
            'gpu': (gpu_decode, gpu_prep, gpu_infer)
        }
    else:
        print("\nGPU not available, skipping GPU preprocessing test.")
        return {'cpu': (cpu_decode, cpu_prep, cpu_infer)}


In [17]:
def task9_fps_stability(video_files: List[str], output_dir: str = "HW1/results", force_rerun: bool = False):
    """Задача 9: Измерение стабильности FPS.

    Args:
        video_files: список путей к видеофайлам
        output_dir: директория для сохранения результатов
        force_rerun: если True, перезапускает измерения даже если есть сохраненные результаты
    """
    os.makedirs(output_dir, exist_ok=True)

    # Создаем уникальный ключ для чекпоинта
    config_hash = hashlib.md5(
        (str(sorted(video_files)) + "16_2_2_4_8_1_2_4_100").encode()
    ).hexdigest()[:8]
    checkpoint_file = f"{output_dir}/task9_checkpoint_{config_hash}.json"

    # Пытаемся загрузить сохраненные результаты
    if not force_rerun and os.path.exists(checkpoint_file):
        try:
            with open(checkpoint_file, 'r') as f:
                saved_data = json.load(f)
                results = [(r['batch_size'], r['prefetch_factor'], r['mean_fps'], r['cv'])
                           for r in saved_data['results']]
                print(f"✓ Loaded checkpoint from {checkpoint_file}")
                print(f"  Previous results: {len(results)} configurations")

                # Строим графики и выводим результаты
                batch_sizes = [2, 4, 8]

                # График FPS
                fig, ax = plt.subplots(figsize=(12, 6))
                for batch_size in batch_sizes:
                    fps_values = [fps for bs, pf, fps, cv in results if bs == batch_size]
                    prefetch_values = [pf for bs, pf, fps, cv in results if bs == batch_size]
                    if fps_values:
                        ax.plot(prefetch_values, fps_values, 'o-', label=f'Batch={batch_size}', linewidth=2)
                ax.set_xlabel('Prefetch Factor', fontsize=12)
                ax.set_ylabel('FPS', fontsize=12)
                ax.set_title('FPS Stability Analysis', fontsize=14)
                ax.legend()
                ax.grid(True, alpha=0.3)
                plt.savefig(f"{output_dir}/task9_fps_stability.png", dpi=150, bbox_inches='tight')
                plt.close()

                # График CV
                fig, ax = plt.subplots(figsize=(12, 6))
                for batch_size in batch_sizes:
                    cv_values = [cv for bs, pf, fps, cv in results if bs == batch_size]
                    prefetch_values = [pf for bs, pf, fps, cv in results if bs == batch_size]
                    if cv_values:
                        ax.plot(prefetch_values, cv_values, 'o-', label=f'Batch={batch_size}', linewidth=2)
                ax.axhline(y=0.05, color='r', linestyle='--', label='Stability threshold (CV=0.05)')
                ax.set_xlabel('Prefetch Factor', fontsize=12)
                ax.set_ylabel('Coefficient of Variation (CV)', fontsize=12)
                ax.set_title('FPS Stability (CV)', fontsize=14)
                ax.legend()
                ax.grid(True, alpha=0.3)
                plt.savefig(f"{output_dir}/task9_fps_cv.png", dpi=150, bbox_inches='tight')
                plt.close()

                # Определение стабильных конфигураций
                stable_configs = [(bs, pf, fps, cv) for bs, pf, fps, cv in results if cv < 0.05]
                print(f"\nStable configurations (CV < 0.05): {len(stable_configs)}")
                for bs, pf, fps, cv in stable_configs:
                    print(f"  Batch={bs}, Prefetch={pf}: FPS={fps:.2f}, CV={cv:.4f}")

                print(f"\nResults saved to {output_dir}/task9_fps_stability.png and task9_fps_cv.png")
                return results
        except Exception as e:
            print(f"Warning: Failed to load checkpoint: {e}. Running fresh measurements...")

    transform = transforms_v2.Compose([
        transforms_v2.ToImage(),
        transforms_v2.Resize((224, 224)),
        transforms_v2.ToDtype(torch.float32, scale=True),
        transforms_v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    batch_sizes = [2, 4, 8]
    prefetch_factors = [1, 2, 4]

    results = []
    total_configs = len(batch_sizes) * len(prefetch_factors)
    config_count = 0

    for batch_size in batch_sizes:
        for prefetch_factor in prefetch_factors:
            config_count += 1
            dataset = VideoDataset(video_files, clip_len=16, stride=2, transform=transform)
            dataloader = DataLoader(
                dataset,
                batch_size=batch_size,
                num_workers=4,
                prefetch_factor=prefetch_factor,
                pin_memory=True
            )

            fps_list = []

            pbar = tqdm(enumerate(dataloader), total=min(100, len(dataloader)),
                       desc=f"  Batch={batch_size}, Prefetch={prefetch_factor} ({config_count}/{total_configs})",
                       leave=False)

            for i, batch in pbar:
                if i >= 100:  # 100 итераций
                    break

                start = time.time()
                # Имитация обработки
                _ = batch.mean()
                elapsed = time.time() - start

                frames_in_batch = batch.shape[0] * batch.shape[1]
                fps = frames_in_batch / elapsed if elapsed > 0 else 0
                fps_list.append(fps)

                # Обновляем описание progress bar
                if fps_list:
                    current_fps = np.mean(fps_list)
                    pbar.set_postfix({'FPS': f'{current_fps:.2f}'})

            if fps_list:
                mean_fps = np.mean(fps_list)
                std_fps = np.std(fps_list)
                cv = std_fps / mean_fps if mean_fps > 0 else float('inf')

                results.append((batch_size, prefetch_factor, mean_fps, cv))

                print(f"  ✓ Batch={batch_size}, Prefetch={prefetch_factor}: "
                      f"FPS={mean_fps:.2f}, CV={cv:.4f} {'✓' if cv < 0.05 else '✗'}")

                # Сохраняем промежуточные результаты
                checkpoint_data = {
                    'results': [
                        {'batch_size': bs, 'prefetch_factor': pf, 'mean_fps': float(fps), 'cv': float(cv)}
                        for bs, pf, fps, cv in results
                    ],
                    'timestamp': time.time()
                }
                with open(checkpoint_file, 'w') as f:
                    json.dump(checkpoint_data, f, indent=2)

    # График FPS
    fig, ax = plt.subplots(figsize=(12, 6))

    for batch_size in batch_sizes:
        fps_values = [fps for bs, pf, fps, cv in results if bs == batch_size]
        prefetch_values = [pf for bs, pf, fps, cv in results if bs == batch_size]
        ax.plot(prefetch_values, fps_values, 'o-', label=f'Batch={batch_size}', linewidth=2)

    ax.set_xlabel('Prefetch Factor', fontsize=12)
    ax.set_ylabel('FPS', fontsize=12)
    ax.set_title('FPS Stability Analysis', fontsize=14)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.savefig(f"{output_dir}/task9_fps_stability.png", dpi=150, bbox_inches='tight')
    plt.close()

    # График CV
    fig, ax = plt.subplots(figsize=(12, 6))

    for batch_size in batch_sizes:
        cv_values = [cv for bs, pf, fps, cv in results if bs == batch_size]
        prefetch_values = [pf for bs, pf, fps, cv in results if bs == batch_size]
        ax.plot(prefetch_values, cv_values, 'o-', label=f'Batch={batch_size}', linewidth=2)

    ax.axhline(y=0.05, color='r', linestyle='--', label='Stability threshold (CV=0.05)')
    ax.set_xlabel('Prefetch Factor', fontsize=12)
    ax.set_ylabel('Coefficient of Variation (CV)', fontsize=12)
    ax.set_title('FPS Stability (CV)', fontsize=14)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.savefig(f"{output_dir}/task9_fps_cv.png", dpi=150, bbox_inches='tight')
    plt.close()

    # Определение стабильных конфигураций
    stable_configs = [(bs, pf, fps, cv) for bs, pf, fps, cv in results if cv < 0.05]

    print(f"\nStable configurations (CV < 0.05): {len(stable_configs)}")
    for bs, pf, fps, cv in stable_configs:
        print(f"  Batch={bs}, Prefetch={pf}: FPS={fps:.2f}, CV={cv:.4f}")

    print(f"\nResults saved to {output_dir}/task9_fps_stability.png and task9_fps_cv.png")
    print(f"Checkpoint saved to {checkpoint_file}")

    return results


## Задача 10: Финальное задание - мини-RT пайплайн


In [18]:
class RealTimePipeline:
    """Near-real-time пайплайн с двухуровневой очередью."""

    def __init__(
        self,
        video_source: str,
        model: Callable,
        frame_queue_size: int = 30,
        clip_queue_size: int = 5,
        clip_len: int = 16,
        stride: int = 2
    ):
        self.video_source = video_source
        self.model = model
        self.frame_queue_size = frame_queue_size
        self.clip_queue_size = clip_queue_size
        self.clip_len = clip_len
        self.stride = stride

        self.frame_queue = queue.Queue(maxsize=frame_queue_size)
        self.clip_queue = queue.Queue(maxsize=clip_queue_size)

        self.running = False
        self.stats = {
            'fps': deque(maxlen=100),
            'latencies': deque(maxlen=100),
            'frame_times': deque(maxlen=100)
        }

    def decode_thread(self):
        """Поток декодирования кадров."""
        if self.video_source.startswith('rtsp://') or self.video_source.startswith('http://'):
            cap = cv2.VideoCapture(self.video_source)
        else:
            container = av.open(self.video_source)
            video_stream = container.streams.video[0]
            cap = None

        frame_buffer = deque(maxlen=self.clip_len * self.stride)

        try:
            if cap is None:
                # PyAV для файлов
                for frame in container.decode(video_stream):
                    if not self.running:
                        break

                    img = frame.to_ndarray(format='rgb24')
                    frame_buffer.append(img)

                    if len(frame_buffer) >= self.clip_len * self.stride:
                        # Формируем клип
                        clip_frames = [frame_buffer[i] for i in range(0, len(frame_buffer), self.stride)][:self.clip_len]
                        clip = np.stack(clip_frames, axis=0)

                        try:
                            self.clip_queue.put(clip, timeout=0.1)
                        except queue.Full:
                            pass  # Пропускаем если очередь полна
            else:
                # OpenCV для потоков
                while self.running:
                    ret, frame = cap.read()
                    if not ret:
                        break

                    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    frame_buffer.append(frame_rgb)

                    if len(frame_buffer) >= self.clip_len * self.stride:
                        clip_frames = [frame_buffer[i] for i in range(0, len(frame_buffer), self.stride)][:self.clip_len]
                        clip = np.stack(clip_frames, axis=0)

                        try:
                            self.clip_queue.put(clip, timeout=0.1)
                        except queue.Full:
                            pass
        finally:
            if cap:
                cap.release()
            if 'container' in locals():
                container.close()

    def process_thread(self):
        """Поток обработки клипов."""
        transform = transforms_v2.Compose([
            transforms_v2.ToImage(),
            transforms_v2.Resize((224, 224)),
            transforms_v2.ToDtype(torch.float32, scale=True),
            transforms_v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        device = 'cuda' if torch.cuda.is_available() else 'cpu'

        while self.running:
            try:
                clip = self.clip_queue.get(timeout=1.0)

                start_time = time.time()

                # Препроцессинг
                # Применяем transform к каждому кадру отдельно
                processed_frames = []
                for frame_idx in range(clip.shape[0]):
                    frame = clip[frame_idx]  # (H, W, 3) numpy array uint8
                    processed_frame = transform(frame)
                    processed_frames.append(processed_frame)
                processed = torch.stack(processed_frames, dim=0)  # (T, C, H, W)

                if device == 'cuda':
                    processed = processed.cuda()
                elif device == 'mps':
                    processed = processed.to('mps')

                # Инференс
                result = self.model(processed)

                latency = time.time() - start_time

                # Обновление статистики
                self.stats['latencies'].append(latency)
                fps = self.clip_len / latency if latency > 0 else 0
                self.stats['fps'].append(fps)

            except queue.Empty:
                continue

    def run(self, duration: float = 30.0):
        """Запуск пайплайна на указанное время."""
        self.running = True

        decode_thread = threading.Thread(target=self.decode_thread, daemon=True)
        process_thread = threading.Thread(target=self.process_thread, daemon=True)

        decode_thread.start()
        process_thread.start()

        time.sleep(duration)
        self.running = False

        decode_thread.join(timeout=5)
        process_thread.join(timeout=5)

    def get_stats(self):
        """Получение статистики."""
        if not self.stats['fps']:
            return None

        fps_array = np.array(self.stats['fps'])
        latencies_array = np.array(self.stats['latencies'])

        mean_fps = np.mean(fps_array)
        p95_latency = np.percentile(latencies_array, 95) * 1000  # мс

        # Jitter как стандартное отклонение латентности
        jitter = np.std(latencies_array) / np.mean(latencies_array) if np.mean(latencies_array) > 0 else 0

        return {
            'mean_fps': mean_fps,
            'p95_latency_ms': p95_latency,
            'jitter': jitter,
            'fps_history': list(fps_array),
            'latency_history': list(latencies_array * 1000)  # мс
        }


In [19]:
def task10_realtime_pipeline(video_file: str, output_dir: str = "HW1/results"):
    """Задача 10: Финальный near-real-time пайплайн."""
    os.makedirs(output_dir, exist_ok=True)

    model = lambda x: x.mean()

    pipeline = RealTimePipeline(
        video_file,
        model,
        frame_queue_size=30,
        clip_queue_size=5,
        clip_len=16,
        stride=2
    )

    print("Running real-time pipeline for 30 seconds...")
    pipeline.run(duration=30.0)

    stats = pipeline.get_stats()

    if stats:
        print("\n" + "="*60)
        print("Real-Time Pipeline Statistics:")
        print("="*60)
        print(f"Mean FPS: {stats['mean_fps']:.2f}")
        print(f"P95 Latency: {stats['p95_latency_ms']:.2f} ms")
        print(f"Jitter: {stats['jitter']:.4f}")

        # Графики
        fig, axes = plt.subplots(2, 1, figsize=(12, 10))

        # FPS over time
        axes[0].plot(stats['fps_history'], linewidth=1, alpha=0.7)
        axes[0].axhline(y=stats['mean_fps'], color='r', linestyle='--', label=f'Mean: {stats["mean_fps"]:.2f}')
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('FPS', fontsize=12)
        axes[0].set_title('FPS Over Time', fontsize=14)
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        # Latency over time
        axes[1].plot(stats['latency_history'], linewidth=1, alpha=0.7)
        axes[1].axhline(y=stats['p95_latency_ms'], color='r', linestyle='--',
                       label=f'P95: {stats["p95_latency_ms"]:.2f} ms')
        axes[1].set_xlabel('Iteration', fontsize=12)
        axes[1].set_ylabel('Latency (ms)', fontsize=12)
        axes[1].set_title('Latency Over Time', fontsize=14)
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig(f"{output_dir}/task10_realtime_stats.png", dpi=150, bbox_inches='tight')
        plt.close()

        print(f"\nResults saved to {output_dir}/task10_realtime_stats.png")

        return stats
    else:
        print("No statistics collected.")
        return None


## Мини-ДЗ: Offline vs Near-Real-Time режимы


In [20]:
def minihw_offline_vs_realtime(video_file: str, output_dir: str = "HW1/results"):
    """Сравнение offline и near-real-time режимов."""
    os.makedirs(output_dir, exist_ok=True)

    model = lambda x: x.mean()

    # Offline режим
    print("Testing offline mode...")
    transform = transforms_v2.Compose([
        transforms_v2.ToImage(),
        transforms_v2.Resize((224, 224)),
        transforms_v2.ToDtype(torch.float32, scale=True),
        transforms_v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    dataset = VideoDataset([video_file], clip_len=16, stride=2, transform=transform)
    dataloader = DataLoader(dataset, batch_size=4, num_workers=4, prefetch_factor=2, pin_memory=True)

    offline_fps_list = []
    offline_latencies = []

    start_time = time.time()
    for i, batch in enumerate(dataloader):
        if i >= 50:
            break

        iter_start = time.time()
        result = model(batch)
        iter_latency = time.time() - iter_start

        frames = batch.shape[0] * batch.shape[1]
        fps = frames / iter_latency if iter_latency > 0 else 0

        offline_fps_list.append(fps)
        offline_latencies.append(iter_latency)

    offline_total_time = time.time() - start_time
    offline_mean_fps = np.mean(offline_fps_list)
    offline_p95_latency = np.percentile(offline_latencies, 95) * 1000
    offline_jitter = np.std(offline_latencies) / np.mean(offline_latencies) if np.mean(offline_latencies) > 0 else 0

    print(f"  Mean FPS: {offline_mean_fps:.2f}")
    print(f"  P95 Latency: {offline_p95_latency:.2f} ms")
    print(f"  Jitter: {offline_jitter:.4f}")

    # Near-Real-Time режим
    print("\nTesting near-real-time mode...")
    pipeline = RealTimePipeline(video_file, model, clip_len=16, stride=2)
    pipeline.run(duration=10.0)

    realtime_stats = pipeline.get_stats()

    if realtime_stats:
        print(f"  Mean FPS: {realtime_stats['mean_fps']:.2f}")
        print(f"  P95 Latency: {realtime_stats['p95_latency_ms']:.2f} ms")
        print(f"  Jitter: {realtime_stats['jitter']:.4f}")

        # Таблица сравнения
        print("\n" + "="*60)
        print("Offline vs Near-Real-Time Comparison:")
        print("="*60)
        print(f"{'Metric':<20} | {'Offline':<15} | {'Near-RT':<15}")
        print("-" * 60)
        print(f"{'Mean FPS':<20} | {offline_mean_fps:>13.2f} | {realtime_stats['mean_fps']:>13.2f}")
        print(f"{'P95 Latency (ms)':<20} | {offline_p95_latency:>13.2f} | {realtime_stats['p95_latency_ms']:>13.2f}")
        print(f"{'Jitter':<20} | {offline_jitter:>13.4f} | {realtime_stats['jitter']:>13.4f}")

        return {
            'offline': {
                'mean_fps': offline_mean_fps,
                'p95_latency_ms': offline_p95_latency,
                'jitter': offline_jitter
            },
            'realtime': realtime_stats
        }

    return None


## Примеры использования

Ниже приведены примеры вызова функций для каждой задачи.


In [21]:
# Укажите путь к вашему видеофайлу
video_path = "tall.mp4"  # Измените на путь к вашему видео
output_dir = "HW1/results"


### Задача 1: Базовый декодер


In [22]:
# Проверка CFR/VFR
video_type, metrics = check_cfr_vfr(video_path)
print(f"Video type: {video_type}")
if metrics.get('avg_frame_rate'):
    print(f"Average frame rate: {metrics['avg_frame_rate']:.2f} fps")

# Чтение клипа
clip = read_clip(video_path, start=0, num_frames=16, stride=2, verbose=True)
print(f"Clip shape: {clip.shape}")

# Визуализация
os.makedirs(output_dir, exist_ok=True)
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(clip[0])
axes[0].set_title('First frame')
axes[0].axis('off')
axes[1].imshow(clip[-1])
axes[1].set_title('Last frame')
axes[1].axis('off')
plt.savefig(f"{output_dir}/task1_frames.png", dpi=150, bbox_inches='tight')
plt.close()
print(f"Frames saved to {output_dir}/task1_frames.png")


Video type: CFR
Average frame rate: 30.00 fps
Read 16 frames. Indices: [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30]
Actual FPS: 30.00
Frame intervals: avg=0.0667s, std=0.0000s
  → CFR detected (constant frame rate)
Clip shape: (16, 1920, 1080, 3)
Frames saved to HW1/results/task1_frames.png


### Задача 3: Параллельная загрузка


In [ ]:
video_files = [video_path]
task3_parallel_loading(video_files, output_dir, force_rerun=False)


Testing num_workers:   0%|          | 0/4 [00:00<?, ?config/s]


Testing with num_workers=1...



  Iterations:   0%|          | 0/5 [00:00<?, ?it/s]

    Epoch 1/5:   0%|          | 0/40 [00:00<?, ?batch/s]

    Epoch 1/5:   2%|▎         | 1/40 [00:06<04:00,  6.17s/batch]

    Epoch 1/5:   5%|▌         | 2/40 [00:12<04:08,  6.53s/batch]

    Epoch 1/5:   8%|▊         | 3/40 [00:15<03:00,  4.88s/batch]

    Epoch 1/5:  10%|█         | 4/40 [00:18<02:30,  4.17s/batch]

    Epoch 1/5:  12%|█▎        | 5/40 [00:23<02:26,  4.19s/batch]

    Epoch 1/5:  15%|█▌        | 6/40 [00:26<02:13,  3.92s/batch]

    Epoch 1/5:  18%|█▊        | 7/40 [00:30<02:09,  3.92s/batch]

    Epoch 1/5:  20%|██        | 8/40 [00:35<02:13,  4.19s/batch]

    Epoch 1/5:  22%|██▎       | 9/40 [00:39<02:06,  4.09s/batch]

    Epoch 1/5:  25%|██▌       | 10/40 [00:43<02:01,  4.05s/batch]

    Epoch 1/5:  28%|██▊       | 11/40 [00:48<02:07,  4.39s/batch]

    Epoch 1/5:  30%|███       | 12/40 [00:52<02:02,  4.37s/batch]

    Epoch 1/5:  32%|███▎      | 13/40 [00:57<01:59,  4.42s/batch]

    Epoch 1/5:  35%|███▌  

  ✓ Mean FPS: 14.51 ± 0.34

Testing with num_workers=2...



  Iterations:   0%|          | 0/5 [00:00<?, ?it/s]

    Epoch 1/5:   0%|          | 0/40 [00:00<?, ?batch/s]

    Epoch 1/5:   2%|▎         | 1/40 [00:04<02:48,  4.31s/batch]

    Epoch 1/5:   5%|▌         | 2/40 [00:04<01:15,  1.98s/batch]

    Epoch 1/5:   8%|▊         | 3/40 [00:10<02:16,  3.68s/batch]

    Epoch 1/5:  10%|█         | 4/40 [00:11<01:34,  2.62s/batch]

    Epoch 1/5:  12%|█▎        | 5/40 [00:15<01:56,  3.33s/batch]

    Epoch 1/5:  15%|█▌        | 6/40 [00:17<01:29,  2.62s/batch]

    Epoch 1/5:  18%|█▊        | 7/40 [00:23<02:02,  3.71s/batch]

    Epoch 1/5:  20%|██        | 8/40 [00:24<01:39,  3.10s/batch]

    Epoch 1/5:  22%|██▎       | 9/40 [00:29<01:53,  3.66s/batch]

    Epoch 1/5:  25%|██▌       | 10/40 [00:31<01:34,  3.14s/batch]

    Epoch 1/5:  28%|██▊       | 11/40 [00:38<01:59,  4.13s/batch]

    Epoch 1/5:  30%|███       | 12/40 [00:40<01:42,  3.66s/batch]

    Epoch 1/5:  32%|███▎      | 13/40 [00:45<01:51,  4.13s/batch]

    Epoch 1/5:  35%|███▌  

  ✓ Mean FPS: 16.67 ± 0.10

Testing with num_workers=4...



  Iterations:   0%|          | 0/5 [00:00<?, ?it/s]

    Epoch 1/5:   0%|          | 0/40 [00:00<?, ?batch/s]

    Epoch 1/5:   2%|▎         | 1/40 [00:09<05:56,  9.14s/batch]

    Epoch 1/5:   8%|▊         | 3/40 [00:10<01:41,  2.76s/batch]

    Epoch 1/5:  10%|█         | 4/40 [00:12<01:28,  2.45s/batch]

    Epoch 1/5:  12%|█▎        | 5/40 [00:21<02:42,  4.63s/batch]

    Epoch 1/5:  15%|█▌        | 6/40 [00:21<01:54,  3.36s/batch]

    Epoch 1/5:  18%|█▊        | 7/40 [00:23<01:38,  2.97s/batch]

    Epoch 1/5:  20%|██        | 8/40 [00:26<01:35,  2.97s/batch]

    Epoch 1/5:  22%|██▎       | 9/40 [00:35<02:24,  4.67s/batch]

    Epoch 1/5:  25%|██▌       | 10/40 [00:38<02:05,  4.20s/batch]

    Epoch 1/5:  28%|██▊       | 11/40 [00:40<01:43,  3.58s/batch]

    Epoch 1/5:  30%|███       | 12/40 [00:43<01:34,  3.38s/batch]

    Epoch 1/5:  32%|███▎      | 13/40 [00:52<02:17,  5.10s/batch]

    Epoch 1/5:  35%|███▌      | 14/40 [00:58<02:16,  5.24s/batch]

    Epoch 1/5:  40%|████ 

  ✓ Mean FPS: 16.36 ± 0.14

Testing with num_workers=8...



  Iterations:   0%|          | 0/5 [00:00<?, ?it/s]

    Epoch 1/5:   0%|          | 0/40 [00:00<?, ?batch/s]

    Epoch 1/5:   2%|▎         | 1/40 [00:18<12:09, 18.70s/batch]

    Epoch 1/5:   5%|▌         | 2/40 [00:19<05:13,  8.26s/batch]

    Epoch 1/5:   8%|▊         | 3/40 [00:21<03:12,  5.20s/batch]

    Epoch 1/5:  10%|█         | 4/40 [00:23<02:28,  4.13s/batch]

    Epoch 1/5:  12%|█▎        | 5/40 [00:24<01:44,  3.00s/batch]

    Epoch 1/5:  15%|█▌        | 6/40 [00:27<01:40,  2.96s/batch]

    Epoch 1/5:  18%|█▊        | 7/40 [00:28<01:15,  2.30s/batch]

    Epoch 1/5:  20%|██        | 8/40 [00:30<01:08,  2.14s/batch]

    Epoch 1/5:  22%|██▎       | 9/40 [00:49<03:48,  7.36s/batch]

    Epoch 1/5:  25%|██▌       | 10/40 [00:50<02:42,  5.40s/batch]

    Epoch 1/5:  28%|██▊       | 11/40 [00:53<02:21,  4.88s/batch]

    Epoch 1/5:  30%|███       | 12/40 [00:58<02:12,  4.74s/batch]

    Epoch 1/5:  32%|███▎      | 13/40 [01:00<01:46,  3.95s/batch]

    Epoch 1/5:  35%|███▌  

### Задача 4: Профилирование


In [24]:
task4_profiling(video_files, output_dir)


Profiling results saved to HW1/results/profiler
View with: tensorboard --logdir=HW1/results/profiler

Time distribution:
  Decode: 0.00 ms (0.0%)
  Preprocess: 0.00 ms (0.0%)
  Infer: 56.63 ms (100.0%)

Ratio L_dec:L_prep:L_inf = 0.00:0.00:56.63


### Задача 5: Prefetch и pinned memory


In [28]:
task5_prefetch_pinned_memory(video_files, output_dir, force_rerun=False)


✓ Loaded checkpoint from HW1/results/task5_checkpoint_ad3ab267.json
  Previous results: 2 configurations

Comparison Results:
Baseline             | FPS:   16.61 ±   0.18 | Jitter: 0.0107
Prefetch             | FPS:   16.64 ±   0.08 | Jitter: 0.0049


[('Baseline', 16.61171399730499, 0.1752038030847143, 0.010728438654780325),
 ('Prefetch', 16.63999409947342, 0.0815015753484083, 0.004886738225426892)]

✓ Loaded checkpoint from HW1/results/task5_checkpoint_ad3ab267.json
  Previous results: 2 configurations

Comparison Results:
Baseline             | FPS:   16.61 ±   0.18 | Jitter: 0.0107
Prefetch             | FPS:   16.64 ±   0.08 | Jitter: 0.0049


[('Baseline', 16.61171399730499, 0.1752038030847143, 0.010728438654780325),
 ('Prefetch', 16.63999409947342, 0.0815015753484083, 0.004886738225426892)]

### Задача 6: Pipeline overlap


In [29]:
task6_pipeline_overlap(video_path, output_dir)


Running sequential pipeline...
  Average latency: 45.96 ms
Running overlapped pipeline...
  Average latency: 187.26 ms

Speedup: 0.25x


(0.04595858573913574, 0.18725702285766602, 0.24543050529041488)

### Задача 7: Аппаратное декодирование


In [30]:
task7_hardware_decoding(video_path, output_dir)


Testing PyAV (CPU)...
  Average time: 1489.52 ms
  FPS: 67.14

Testing decord (CPU)...
  Average time: 2103.45 ms
  FPS: 47.54

Decoding Performance Comparison:
Method               | Time (ms)       | FPS       
------------------------------------------------------------
PyAV (CPU)           |       1489.52 |    67.14
decord (CPU)         |       2103.45 |    47.54


[('PyAV (CPU)', np.float64(1489.5249366760254), np.float64(67.13549906935877)),
 ('decord (CPU)',
  np.float64(2103.452253341675),
  np.float64(47.54089370991607))]

### Задача 8: GPU препроцессинг


In [31]:
task8_gpu_preprocessing(video_files, output_dir)


Testing CPU preprocessing...
  Decode: 868.82 ms
  Preprocess: 250.73 ms
  Infer: 0.75 ms

GPU not available, skipping GPU preprocessing test.


{'cpu': (np.float64(868.8176472981771),
  np.float64(250.72832902272543),
  np.float64(0.7547537485758463))}

### Задача 9: Стабильность FPS


In [32]:
task9_fps_stability(video_files, output_dir, force_rerun=False)


  ✓ Batch=2, Prefetch=1: FPS=3771.65, CV=0.4536 ✗


  ✓ Batch=2, Prefetch=2: FPS=4020.43, CV=0.5284 ✗


  ✓ Batch=2, Prefetch=4: FPS=3931.64, CV=0.4251 ✗


  ✓ Batch=4, Prefetch=1: FPS=4136.16, CV=0.5082 ✗


  ✓ Batch=4, Prefetch=2: FPS=4177.80, CV=0.4847 ✗


  ✓ Batch=4, Prefetch=4: FPS=4491.75, CV=0.4559 ✗


  ✓ Batch=8, Prefetch=1: FPS=4858.04, CV=0.5668 ✗


  ✓ Batch=8, Prefetch=2: FPS=4091.07, CV=0.6742 ✗


  ✓ Batch=8, Prefetch=4: FPS=4363.37, CV=0.4327 ✗

Stable configurations (CV < 0.05): 0

Results saved to HW1/results/task9_fps_stability.png and task9_fps_cv.png
Checkpoint saved to HW1/results/task9_checkpoint_529778e0.json


[(2, 1, np.float64(3771.653101660512), np.float64(0.45362100309954556)),
 (2, 2, np.float64(4020.426204167688), np.float64(0.5283626868904134)),
 (2, 4, np.float64(3931.6377857399243), np.float64(0.42514938688412274)),
 (4, 1, np.float64(4136.156102634188), np.float64(0.5082349430170562)),
 (4, 2, np.float64(4177.804975984009), np.float64(0.4846990608769647)),
 (4, 4, np.float64(4491.748094871177), np.float64(0.4558575070224024)),
 (8, 1, np.float64(4858.041854039516), np.float64(0.5668106159659145)),
 (8, 2, np.float64(4091.0710973508003), np.float64(0.6741749905036375)),
 (8, 4, np.float64(4363.3691423199), np.float64(0.4327062252079931))]

### Задача 10: Real-time пайплайн


In [33]:
task10_realtime_pipeline(video_path, output_dir)


Running real-time pipeline for 30 seconds...

Real-Time Pipeline Statistics:
Mean FPS: 54.17
P95 Latency: 518.14 ms
Jitter: 0.2791

Results saved to HW1/results/task10_realtime_stats.png


{'mean_fps': np.float64(54.17195574046172),
 'p95_latency_ms': np.float64(518.1372880935669),
 'jitter': np.float64(0.27908168349221457),
 'fps_history': [np.float64(31.36996855470859),
  np.float64(38.56687192719812),
  np.float64(47.641330174702034),
  np.float64(56.65882384725921),
  np.float64(58.207484441746),
  np.float64(57.91072915596984),
  np.float64(56.54496416062041),
  np.float64(59.13019083048001),
  np.float64(26.98612951389964),
  np.float64(31.226424023023537),
  np.float64(57.37261627320463),
  np.float64(58.942912853303355),
  np.float64(30.954672846328272),
  np.float64(30.035502556037233),
  np.float64(33.41893492833307),
  np.float64(31.117092387473942),
  np.float64(31.049828623883094),
  np.float64(41.595793119563744),
  np.float64(55.396265414595774),
  np.float64(55.5141736842976),
  np.float64(59.663195515627294),
  np.float64(59.99415692970615),
  np.float64(59.480860062947436),
  np.float64(56.07911207822964),
  np.float64(59.91606074025399),
  np.float64(6

### Мини-ДЗ: Offline vs Real-time


In [34]:
minihw_offline_vs_realtime(video_path, output_dir)


Testing offline mode...
  Mean FPS: 3931.29
  P95 Latency: 26.18 ms
  Jitter: 0.3391

Testing near-real-time mode...
  Mean FPS: 53.14
  P95 Latency: 516.94 ms
  Jitter: 0.2862

Offline vs Near-Real-Time Comparison:
Metric               | Offline         | Near-RT        
------------------------------------------------------------
Mean FPS             |       3931.29 |         53.14
P95 Latency (ms)     |         26.18 |        516.94
Jitter               |        0.3391 |        0.2862


{'offline': {'mean_fps': np.float64(3931.2908981336936),
  'p95_latency_ms': np.float64(26.184237003326412),
  'jitter': np.float64(0.3390790747581463)},
 'realtime': {'mean_fps': np.float64(53.13528962726998),
  'p95_latency_ms': np.float64(516.9436693191528),
  'jitter': np.float64(0.28616962030625376),
  'fps_history': [np.float64(35.57055316830282),
   np.float64(57.672590145563234),
   np.float64(58.699513059572574),
   np.float64(56.14564146498179),
   np.float64(60.157108537416185),
   np.float64(59.9800009652734),
   np.float64(57.99646017699115),
   np.float64(57.86573817789252),
   np.float64(60.32550254124695),
   np.float64(58.89904404820834),
   np.float64(59.37686766900928),
   np.float64(57.10158314713245),
   np.float64(59.33738707712222),
   np.float64(58.020979897651024),
   np.float64(60.80888145465353),
   np.float64(35.59617521613466),
   np.float64(31.00477022308936),
   np.float64(29.03428708391488),
   np.float64(30.907412976727628),
   np.float64(33.20141792392